# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring a dataset using the `mlcroissant` library, referencing all entities by their `@id` attributes for clarity and reproducibility.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs. All references are by `@id`.

In [ ]:
# List all record sets and their fields using metadata
record_sets = metadata.record_set
if not record_sets:
    # Dataset may not list record_set directly - try to retrieve from dataset
    record_sets = dataset.record_sets()

print("Available record sets:")
for rs in record_sets:
    print(f"- @id: {rs['@id']} | Name: {rs.get('name', 'N/A')}")
    if 'field' in rs:
        print("  Fields:")
        for f in rs['field']:
            print(f"    - @id: {f['@id']} | Name: {f.get('name', 'N/A')} | DataType: {f.get('dataType', 'N/A')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from all record sets
dataframes = {}

# Use the first record set for demonstration
record_sets_all = record_sets if record_sets else dataset.record_sets()
record_set_ids = [rs['@id'] for rs in record_sets_all]

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded '@id': {record_set_id} with columns: {df.columns.tolist()}")
    except Exception as e:
        print(f"Failed to load record set @{record_set_id}: {e}")

# For analysis, select the first available record set
main_record_set_id = record_set_ids[0] if record_set_ids else list(dataframes.keys())[0]
print(f"\nPreview of data from record set '@id': {main_record_set_id}")
if main_record_set_id in dataframes:
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. Operations like removing outliers, transforming data distributions, or grouping data may be included.

In [ ]:
# For EDA, let's select 'GAD-7 score' as a numeric field if available
# Use @id to reference; from Croissant schema, this may be like 'gad7_score', but let's scan the field ids

df = dataframes[main_record_set_id]

# Find numeric fields by scanning columns
numeric_fields = [col for col in df.columns if 'score' in col.lower() or df[col].dtype in [int,float]]
if numeric_fields:
    numeric_field_id = numeric_fields[0]
else:
    numeric_field_id = df.columns[0]  # fallback

threshold = 10
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records where '{numeric_field_id}' > {threshold}:")
display(filtered_df.head())

# Normalizing selected numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized '{numeric_field_id}' for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try grouping by a key demographic field, e.g. 'village', referenced by its @id
group_field_id = None
for col in df.columns:
    if 'village' in col.lower():
        group_field_id = col
        break

if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Grouped data by '{group_field_id}':")
    display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of selected numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
plt.title(f"Distribution of '{numeric_field_id}'")
plt.xlabel(numeric_field_id)
plt.ylabel("Frequency")
plt.show()

# If possible, visualize normalized score per village
if group_field_id:
    plt.figure(figsize=(10,6))
    sns.barplot(x=group_field_id, y=f"{numeric_field_id}_normalized", data=filtered_df)
    plt.title(f"Normalized '{numeric_field_id}' by '{group_field_id}'")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR² Mental Health Survey Dataset provides rich information on psychological symptoms across Kilifi County, Kenya.
- Using Croissant's schema and `mlcroissant`, we loaded data referencing every entity by its `@id`, ensuring robust, reproducible analysis pipelines.
- Numeric fields such as assessment scores (e.g., GAD-7, PHQ-9, PSQ) can be filtered, normalized, and grouped to explore patterns such as symptom distribution by demographics (e.g., village).
- Visualizations help reveal potential insights in community mental health, supporting research and public health interventions.

For further analysis, the dataset's schema enables clear mapping between metadata, data values, and potential for automation in downstream ML workflows.